In [29]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import PromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
# from langchain_community.llms import HuggingFacePipeline # For older versions of langchain, use the above import. For newer versions, use the below import.
from langchain_huggingface import HuggingFacePipeline

2.7.0+cu128
True
12.8
NVIDIA GeForce RTX 3080 Ti


#### google/flan-t5-base

In [2]:
#Using smaller model
generator = pipeline("text2text-generation", model="google/flan-t5-base")

def get_completion(prompt):
    response = generator(prompt, max_new_tokens=100, do_sample=False)
    return (response[0]["generated_text"].strip())

print(get_completion("What is a DEFI in context of crypto world"))
#Using different variant i.e larger model, to run remove """ """

e:\github\GenAI_HF_OpenAI\Hugging_Face_OpenAI_Azure\Code_Examples\.venv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


crypto cryptography


#### mistralai/Mistral-7B-Instruct-v0.1

In [3]:
#from transformers import pipeline
#Note**Access to model mistralai/Mistral-7B-Instruct-v0.1 is restricted. You must have access to it and be
#authenticated to access it. If yes, then we can use the code below.
#Note ** this will download large tensors,configs etc.. for this model, thus to run remove """ """ & then run

# generator = pipeline("text-generation", model="mistralai/Mistral-7B-Instruct-v0.1")


# def get_completion(prompt):
#     # For instruction-tuned models, prepend with an instruction-style format
#     instruction = f"<s>[INST] {prompt} [/INST]"
#     response = generator(instruction, max_new_tokens=100, do_sample=False)
#     return response[0]["generated_text"].split("[/INST]")[-1].strip()

# print(get_completion("What is defi in context of crypto world?"))

#### tiiuae/falcon-7b-instruct

In [4]:
##using Another heavier model
#Note ** this will download large tensors,configs etc.. for this model, thus to run remove """ """ & then run

# Load a text-generation pipeline with an instruction-tuned model
# generator = pipeline("text-generation", model="tiiuae/falcon-7b-instruct")

# def get_completion(prompt):
#     # For instruction-tuned models, prepend with an instruction-style format
#     instruction = f"<s>[INST] {prompt} [/INST]"
#     response = generator(instruction, max_new_tokens=100, do_sample=False)
#     return response[0]["generated_text"].split("[/INST]")[-1].strip()

# print(get_completion("What is defi in context of crypto world?"))

In [5]:
#Back to Chain Approach & using templates
# Load the model and tokenizer locally
model_name = "google/flan-t5-large"  # You can also use "google/flan-t5-xl"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

client = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100, 
    torch_dtype=torch.float32,  # Uses lower precision for efficiency
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available
)

#### Chain approach

In [6]:
template2 = " Please write a {length} review, of the book {book_title}. "
input_variables2 = [ "length", "book_title" ]
prompt = PromptTemplate(
    input_variables=input_variables2,
    template=template2
)

#To check
formatted_prompt = prompt.format(length = "short", book_title = "House Of Dragon")
print(formatted_prompt)

 Please write a short review, of the book House Of Dragon. 


In [7]:
llm = HuggingFacePipeline(pipeline=client)

chain_new = prompt | llm
response = chain_new.invoke({
    "length": "short",
    "book_title": "House of Dragon"
})
print(response)

a spooky tale of dragons and dragon-like creatures


#### Function approach

In [8]:
#Using function as defined above
formatted_prompt = prompt.format(length = "short", book_title = "House Of Dragon")
def get_completion(prompt):
    response = client(prompt, max_new_tokens=100, do_sample=False)
    return (response[0]["generated_text"].strip())
response = get_completion(formatted_prompt)
print("AI Response:")
print(type(response))
print(response)

AI Response:
<class 'str'>
a spooky tale of dragons and dragon-like creatures


#### Switching to usage of bigger LLMs deployed on endpoints

In [9]:
import os

import openai
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv("../.env")

# Initialize client once
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

deployment_name = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

#or
#Using Langchain Equivalent:

"""
from langchain_openai import AzureChatOpenAI
from dotenv import load_dotenv
load_dotenv("../.env")

client = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT"),
    temperature=0,
)
llm.invoke("Explain transformers in simple terms in 25 words").content
"""

'\nfrom langchain_openai import AzureChatOpenAI\nfrom dotenv import load_dotenv\nload_dotenv("../.env")\n\nclient = AzureChatOpenAI(\n    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),\n    api_key=os.getenv("AZURE_OPENAI_API_KEY"),\n    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),\n    deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT"),\n    temperature=0,\n)\nllm.invoke("Explain transformers in simple terms in 25 words").content\n'

In [10]:
#Creating function
def get_completion(prompt, deployment_name=deployment_name):
    """Get a chat completion from Azure OpenAI.
    Args:
        prompt (str): User input prompt.
        deployment_name (str): The deployment name you gave your model in Azure portal.
    Returns:
        dict: Full response object, or error dict.
    """
    try:
        messages = [{"role": "user", "content": prompt}]
        response = client.chat.completions.create(
            model=deployment_name,    # <-- This is the "deployment name" not the raw model name
            messages=messages,
            temperature=0.1,
            top_p=0.8,
            max_tokens=512
        )
        #return response.model_dump()  # Return the full response as dict
        # Extract just the assistant's reply
        return response.choices[0].message.content
    except Exception as e:
        return {"error": str(e)}

In [12]:
prompt = "Explain transformers in simple terms in 25 words"
response = get_completion(prompt)
response

'Transformers are models that understand language by focusing on important words in a sentence, enabling machines to translate, summarize, and generate text effectively.'

In [13]:
#If using get_completion() based on gpt model as defined above, then we can
"""
response = get_completion(formatted_prompt)
print("AI Response:")
print(type(response))
print(response.keys())

response['choices'][0]['message']['content']"""

'\nresponse = get_completion(formatted_prompt)\nprint("AI Response:")\nprint(type(response))\nprint(response.keys())\n\nresponse[\'choices\'][0][\'message\'][\'content\']'

#### Formatting
* Jinja Template example 1

In [16]:
#Jinja Template example
jinja2_template = "Give me an {{ adjective }} fact about {{ topic }}"

prompt = PromptTemplate.from_template(jinja2_template, template_format = "jinja2")

user_question = prompt.format(adjective="interesting", topic="space exploration")
print(user_question)


Give me an interesting fact about space exploration


In [17]:
response = get_completion(user_question)
print("AI Response:")
print(response)

AI Response:
Sure! Did you know that the Voyager 1 spacecraft, launched in 1977, is the farthest human-made object from Earth? As of now, it has traveled over 14 billion miles away and is currently in interstellar space, sending back data about the environment beyond our solar system. It's expected to keep communicating with Earth until around 2025!


* Jinja Template example 2

In [22]:
# Example where string prompt template would not work
# Define the prompt template

jinja2_template = """

Dear {{ name }},
{% if age < 18 %}
You are invited to our kids' event with activities such as face painting, bouncy castles, and clown shows.
{% elif age < 65 %}
You are invited to our adult event with activities like live music, wine tasting, and art workshops.
{% else %}
You are invited to our senior event with activities including book clubs, chess tournaments, and tea dances.
{% endif %}
Sincerely,
Event Organizer

Write the mail in 200 words
"""

In [24]:
prompt = PromptTemplate.from_template(jinja2_template, template_format="jinja2")

# Format the prompt with specific values for 'action', 'group', and 'time_period'
argument_prompt = prompt.format(name="John Doe", age=12)

In [25]:
response = get_completion(argument_prompt)
print("AI Response:")
print(response)

AI Response:
Subject: Invitation to Our Exciting Kids' Event!

Dear John Doe,

We are delighted to invite you and your family to our upcoming kids' event, designed to provide a fun-filled day for children of all ages. This special occasion promises a variety of engaging activities that will keep the little ones entertained and create wonderful memories.

Our event will feature exciting face painting sessions where children can transform into their favorite characters or animals with the help of talented artists. Additionally, we have arranged for several bouncy castles, offering a safe and thrilling environment for kids to jump, play, and expend their energy. To add to the fun, there will be lively clown shows filled with laughter, magic tricks, and interactive performances that are sure to captivate the young audience.

We believe this event is a fantastic opportunity for children to socialize, explore their creativity, and enjoy a day packed with joy and excitement. Parents can relax

* f-string

In [18]:
#Using f-string (example)
fstring_template = "Here is a brief summary for the book titled '{book_title}':"
book_title = "The Great Gatsby"
prompt = fstring_template.format(book_title=book_title)

In [20]:
#Testing a dummy function
def get_book_summary(prompt):
    return "It's a novel about love, wealth, and aspiration, set in the Roaring '20s."

summary = get_book_summary(prompt)
print(summary)

It's a novel about love, wealth, and aspiration, set in the Roaring '20s.


In [21]:
#Using function that invokes the LLM
response = get_completion(prompt)
print("AI Response:")
print(response)

AI Response:
Sure! Please provide the summary you'd like me to review or help with.


* Langchain templates & chat mode

In [26]:
simple_prompt = "The {subject} is strong in this one."
human_prompt = "Summarize our conversation so far in {word_count} words."

In [30]:
simple_message_template = HumanMessagePromptTemplate.from_template(simple_prompt)
human_message_template = HumanMessagePromptTemplate.from_template(human_prompt)

chat_prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="conversation"),
    simple_message_template,
    human_message_template
])

In [31]:
human_message = HumanMessage(content="What's the best way to learn a new language?")
ai_message = AIMessage(content="""\
1. Immerse yourself in the language: Try to use the language in your daily life as much as possible.
2. Practice regularly: Consistency is key when learning a new language.
3. Use language learning apps: There are many apps that can help you learn a new language in a fun and engaging way.\
""")

In [33]:
conversation = chat_prompt.format_prompt(
    conversation=[human_message, ai_message],
    subject="Force",
    word_count="10"
).to_messages()

print(conversation)

[HumanMessage(content="What's the best way to learn a new language?", additional_kwargs={}, response_metadata={}), AIMessage(content='1. Immerse yourself in the language: Try to use the language in your daily life as much as possible.\n2. Practice regularly: Consistency is key when learning a new language.\n3. Use language learning apps: There are many apps that can help you learn a new language in a fun and engaging way.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='The Force is strong in this one.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Summarize our conversation so far in 10 words.', additional_kwargs={}, response_metadata={})]


In [34]:
simple_prompt = "The {subject} is fascinating to study."
human_prompt = "Summarize our conversation so far in {word_count} words."

In [35]:
human_message = HumanMessage(content="What's happens inside a black hole")
ai_message = AIMessage(content="""\
1. Inside black hole gariivty is zero way.\
""")

In [36]:
conversation = chat_prompt.format_prompt(
    conversation=[human_message, ai_message],
    subject="Black Hole",
    word_count="10"
).to_messages()

In [37]:
prompt_string = conversation
print("Formatted Prompt:")
print(prompt_string)

Formatted Prompt:
[HumanMessage(content="What's happens inside a black hole", additional_kwargs={}, response_metadata={}), AIMessage(content='1. Inside black hole gariivty is zero way.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='The Black Hole is strong in this one.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Summarize our conversation so far in 10 words.', additional_kwargs={}, response_metadata={})]


In [38]:
print(type(prompt_string))

<class 'list'>


#### Using Langchain templates & styles

In [39]:
#Using style
#Modify parameters like **customer_style** and **customer_email**
#to influence the tone and formality of the generated responses.

template_string = """Translate the text that is delimited by triple backticks into a style
that is {style}. text: ```{text}```"""

# Style and email input
customer_style = "American English in a casual tone"
customer_email = """
I'm super excited about the new gaming console I bought! It arrived in just 2 days and I've been playing non-stop. Totally worth the price!
"""

In [40]:
prompt = template_string.format(style=customer_style, text=customer_email)

In [41]:
instruction_prompt = f"<s>[INST] {prompt} [/INST]"

In [ ]:
#Using generator based on google/flan-t5-base
response = generator(instruction_prompt, max_new_tokens=150, do_sample=False)

In [43]:
response

[{'generated_text': " I'm super excited about the new gaming console I bought! It arrived in just 2 days and I've been playing non-stop. Totally worth the price!  [/INST]"}]

In [44]:
generated_text = response[0]['generated_text'].split("[/INST]")[-1].strip()

print(response[0])

{'generated_text': " I'm super excited about the new gaming console I bought! It arrived in just 2 days and I've been playing non-stop. Totally worth the price!  [/INST]"}


In [45]:
template_string = """Translate the text that is delimited by triple backticks into a style that is {style}.
text: ```{text}```"""

prompt_template = ChatPromptTemplate.from_template(template_string)

messages = prompt_template.format_messages(
    style="Scottish English in a professional tone",
    text="I'm super excited about the new gaming console & new game!"
)

#Using generator based on google/flan-t5-base
response = generator(prompt,max_new_tokens=150, do_sample=False)
print(response)

[{'generated_text': " I'm super excited about the new gaming console I bought! It arrived in just 2 days and I've been playing non-stop. Totally worth the price!"}]
